# From Discrete to Continuous Distributions

This notebook makes the leap from discrete to **continuous** probability distributions, using the call-center waiting time example: why the probability of any *exact* value is zero, how thinking in **windows** fixes the problem, and how ever-finer windows converge to a smooth curve whose **area is 1**. It is based on the sixth video of Lesson 2 (Probability Distributions).


## 1. Introduction

Discrete random variables have events that form a **list**: 0, 1, 2, or 3 heads; the population of a town. Continuous ones live on an **interval**: your wait time on a tech-support call can be 1 minute, but also 1.01, or 1.22375... — the values cannot be listed.

Try to plot call-time probabilities as bars at 1, 2, 3 minutes: what about 1.01? 2.43? There are infinitely many values *everywhere*, and since all the "heights" must sum to 1, each individual one is forced smaller and smaller — down to zero.

Nothing went wrong. This distribution is **fundamentally different**.

After completing this notebook you will be able to:

- Explain why $P(X = \text{exact value}) = 0$ for a continuous random variable.
- Describe continuous distributions through **window (interval) probabilities**.
- See the density curve as the limit of ever-finer histograms, with **area under the curve = 1**.


## 2. Theory

### The probability of an exact value is zero

What is the probability that a call takes exactly one minute — 1.000000... forever? **Zero.** There are uncountably many possible durations (a whole interval), so no single point can carry positive probability.

### Windows instead of points

Ask instead: what is the probability that the call lasts **between** 0 and 1 minutes? Between 1 and 2? These window probabilities are perfectly meaningful, and plotting them gives an honest discrete distribution whose heights (the blue areas) sum to 1.

### Refining the windows

Split the windows into 30-second bins, then 15-second bins — more granular information each time, still a valid discrete distribution. Doing this **infinitely many times** leaves a curve: infinitely many, infinitely skinny bars.

### The continuous condition

The discrete rule "heights sum to 1" becomes, in the limit:

$$
\text{area under the curve} = 1
$$

That curve is a continuous probability distribution (its formal name, the *probability density function*, comes in the next videos).


## 3. Worked Example: the call-center distribution

Assume calls last between 0 and 5 minutes, with most calls taking 1-3 minutes. With 1-minute windows we might have:

| Window (min) | Probability |
|---|---|
| 0-1 | 0.15 |
| 1-2 | 0.35 |
| 2-3 | 0.30 |
| 3-4 | 0.15 |
| 4-5 | 0.05 |

Heights sum to 1, and the message is readable: the majority of calls take 1-3 minutes, few run to 5. Refining to 30-second or 15-second windows keeps the total at 1 while showing more detail — and the limit is the smooth curve.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

# A population of call durations between 0 and 5 minutes, bulk at 1-3.
# (Gamma-shaped, clipped to 5 minutes.)
calls = rng.gamma(shape=4.0, scale=0.55, size=200_000)
calls = calls[calls <= 5]
print(f"{len(calls):,} simulated calls, "
      f"mean {calls.mean():.2f} min, max {calls.max():.2f} min")

# numpy compatibility: trapezoid was renamed from trapz in numpy 2.0
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz


### Exact values have probability zero


In [ ]:
print("Calls lasting EXACTLY 1.0 minute:", (calls == 1.0).sum())
print("Calls between 0.95 and 1.05 min:  ", ((calls > 0.95) & (calls < 1.05)).sum())
print("-> points carry no probability; windows do.")


### Window probabilities sum to 1, at any granularity


In [ ]:
for width in [1.0, 0.5, 0.25]:
    edges = np.arange(0, 5 + width, width)
    probs = np.histogram(calls, bins=edges)[0] / len(calls)
    print(f"window {width:4} min: {len(probs):2d} windows, "
          f"probabilities sum to {probs.sum():.4f}")


## 5. Visualizations

### The video's refinement sequence

1-minute windows → 30 seconds → 15 seconds → the limiting curve. The blue area is always exactly 1.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6), sharey=True)

for ax, width, title in [(axes[0], 1.0, "1-minute windows"),
                         (axes[1], 0.5, "30-second windows"),
                         (axes[2], 0.25, "15-second windows")]:
    edges = np.arange(0, 5 + width, width)
    density = np.histogram(calls, bins=edges, density=True)[0]
    ax.bar(edges[:-1], density, width=width, align="edge",
           color="#4e79a7", edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel("Call duration (min)")

# The 'infinitely fine' limit: a smooth density estimate
xs = np.linspace(0, 5, 400)
fine = np.histogram(calls, bins=200, range=(0, 5), density=True)[0]
centers = np.linspace(0, 5, 200, endpoint=False) + 5 / 400
axes[3].fill_between(centers, fine, color="#4e79a7", alpha=0.85)
axes[3].set_title("the limit: a curve")
axes[3].set_xlabel("Call duration (min)")

axes[0].set_ylabel("density (area = probability)")
plt.suptitle("Refining windows forever turns bars into a continuous distribution")
plt.tight_layout()
plt.show()


### Area = 1, and window probability = area of a slice

The shaded slice is $P(1 \leq X \leq 2)$ — probability lives in areas now, not in heights.


In [ ]:
density, edges = np.histogram(calls, bins=200, range=(0, 5), density=True)
centers = (edges[:-1] + edges[1:]) / 2

plt.figure(figsize=(9, 5))
plt.plot(centers, density, color="#2c5f8a", linewidth=2)
mask = (centers >= 1) & (centers <= 2)
plt.fill_between(centers[mask], density[mask], color="#e15759", alpha=0.7,
                 label=f"P(1 ≤ X ≤ 2) ≈ {((calls >= 1) & (calls <= 2)).mean():.2f}")
plt.fill_between(centers, density, color="#4e79a7", alpha=0.15)
total = np.trapezoid(density, centers)
plt.title(f"Total area under the curve: {total:.3f} — the continuous version of 'sums to 1'")
plt.xlabel("Call duration (min)")
plt.ylabel("Density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 6. Exercises

### Basic

1. Classify: (a) number of calls a center receives in an hour, (b) duration of one call, (c) number of heads in 20 flips, (d) height of a person.
2. For a continuous random variable, what is $P(X = 2.5)$ exactly? What kind of question should you ask instead?

### Intermediate

3. Using the 1-minute window table above, what is the probability a call lasts **more than 3 minutes**?
4. If you halve the window width, roughly what happens to each bar's *probability*? What happens to its *density* (probability ÷ width)? This is why the y-axis switches to density.

### Advanced

5. With the simulated `calls` data, compute $P(1 \leq X \leq 2)$ three ways: fraction of samples, sum of 15-second window probabilities, and area under the 200-bin density curve. Confirm all three agree.


## 7. Solutions


In [ ]:
# Exercise 1
print("1.  (a) discrete, (b) continuous, (c) discrete, (d) continuous.")

# Exercise 2
print("2.  P(X = 2.5) = 0. Ask for a window: P(2.4 <= X <= 2.6), etc.")

# Exercise 3
print("3.  P(X > 3) = 0.15 + 0.05 = 0.20")

# Exercise 4
print("4.  Each probability roughly halves (finer slices), but density stays "
      "roughly constant — density is what converges to the curve.")

# Exercise 5
frac = ((calls >= 1) & (calls <= 2)).mean()

edges = np.arange(0, 5.25, 0.25)
probs = np.histogram(calls, bins=edges)[0] / len(calls)
window_sum = probs[4:8].sum()          # windows 1.00-1.25 ... 1.75-2.00

density, d_edges = np.histogram(calls, bins=200, range=(0, 5), density=True)
centers = (d_edges[:-1] + d_edges[1:]) / 2
mask = (centers >= 1) & (centers <= 2)
area = np.trapezoid(density[mask], centers[mask])

print(f"5.  fraction: {frac:.4f}, window sum: {window_sum:.4f}, area: {area:.4f}")


## 8. Conclusion

- Discrete events form a **list**; continuous events fill an **interval** — and no interval can be listed.
- For continuous variables, **every exact value has probability zero**; meaningful questions are about **windows**.
- Refining windows forever turns histograms into a smooth curve; the rule "heights sum to 1" becomes "**area under the curve is 1**."
- Probability now lives in **areas of slices** — the formal machinery for this (the probability density function) is next.
